# Inspect and prune the starved classes

Four classes are too small to learn from, and some of what little they have is not
usable — detached legs, loose body segments, or larvae mixed with adults.

| class | specimens | AP | note |
|---|---|---|---|
| Raphidioptera | 2 | 0.000 | one adult, one larva |
| Siphonaptera | 6 | 0.101 | not viewable on the BIOSCAN site — cutout is the only view |
| Zoraptera | 10 | 0.314 | looks fine on the site; check whether the *cutout* is |
| Odonata | 17 | 0.019 | fragments plus a nymph/adult mix |

Every specimen is shown at full size with its `processid`, so you can read off the
ones to drop. Where the original BIOSCAN photograph is reachable it is shown beside
the cutout — that distinguishes **a bad source image** (a photograph of a leg) from
**a bad segmentation** (a good photograph the pipeline mangled). The two need
different fixes.

Nothing is deleted. The notebook writes a blocklist of `processid`s that the
generator filters against, so a decision can be reversed without re-cutting.

**Runs on Gaia**, CPU only.


In [ ]:
import os
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

BASE_DIR    = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))
CUTOUT_DIR  = BASE_DIR / "DATASET_20000" / "cutouts"
CUTOUT_META = CUTOUT_DIR / "cutouts_metadata.csv"
OUT_DIR     = BASE_DIR / "PRUNE_REVIEW"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Where the original BIOSCAN photographs live, if they are on this machine. The
# panel is skipped cleanly when they are not — the cutouts alone still work.
BIOSCAN_GLOBS = ["**/bioscan*/**/{pid}.jpg", "**/bioscan*/**/{pid}.png",
                 "**/images/**/{pid}.jpg", "**/{pid}.jpg"]

# Training labels. Diplopoda and Chilopoda are CLASSES, not orders — the specimen
# table records the finer rank, so they only resolve through the rollup below.
# The myriapods are no longer one label each. Chilopoda splits into Lithobiomorpha
# (82) and Scutigeromorpha (31) — both above any count that has worked here, and a
# house centipede looks nothing like a stone centipede. Diplopoda keeps only
# Polyxenida, so it is renamed to what it actually contains.
#
# "Diplopoda" and "Chilopoda" remain below only as holding pens for the sub-orders
# being dropped, so the blocklist can still see them.
CLASSES = ["Raphidioptera", "Siphonaptera", "Zoraptera", "Odonata",
           "Polyxenida", "Lithobiomorpha", "Scutigeromorpha",
           "Diplopoda", "Chilopoda"]

TAXON_ROLLUP = {
    "Julida": "Diplopoda", "Chordeumatida": "Diplopoda", "Polydesmida": "Diplopoda",
    "Geophilomorpha": "Chilopoda",
    # Polyxenida, Lithobiomorpha and Scutigeromorpha are now classes in their own
    # right, so they are deliberately absent from this map.
}

# Whole sub-orders to remove. A class label covering morphologically unrelated
# animals is asking one detector head to learn several appearances at once; cutting
# it back to the dominant sub-order trades count for coherence.
DROP_SUBORDERS = {
    "Julida":         "Diplopoda — keeping only Polyxenida",
    "Chordeumatida":  "Diplopoda — keeping only Polyxenida",
    "Polydesmida":    "Diplopoda — keeping only Polyxenida",
    "Geophilomorpha": "Chilopoda — too few, and unlike the rest of the class",
    # Classes removed from the taxonomy entirely. Blocking their specimens keeps
    # them out of the trays as well — otherwise they would still be pasted, and an
    # unlabelled insect on a tray teaches the model that insects are background.
    "Raphidioptera":  "class dropped — 2 specimens, one adult and one larva",
    "Siphonaptera":   "class dropped — 6 specimens, below anything that has worked",
}
MIN_KEEP = 14        # below this a class is not learnable — Phasmatodea manages
                     # 0.59 on 14 clean specimens, Zygentoma 0.63 on 16

# Already identified from the BIOSCAN site. Shown ringed in the contact sheets so
# you can confirm them against the cutout before committing.
PRE_FLAGGED = {
    "CISSB6595-23": "Odonata — flagged from BIOSCAN site",
    "ECORG053-21":  "Odonata — flagged from BIOSCAN site",
    "ECORG054-21":  "Odonata — flagged from BIOSCAN site",
    "ECORG684-22":  "Odonata — flagged from BIOSCAN site",
    "LPASA1616-22": "Raphidioptera — flagged from BIOSCAN site",
}

cut = pd.read_csv(CUTOUT_META)
cut.columns = [c.strip() for c in cut.columns]
assert "order" in cut.columns, f"no order column: {list(cut.columns)}"

# `order` here is the FINE rank (Julida, Polyxenida, ...). `label` is what the model
# is actually trained on, after rolling the myriapod orders up into their classes.
cut["fine_order"] = cut["order"].astype(str)
cut["label"] = cut["fine_order"].map(lambda o: TAXON_ROLLUP.get(o, o))
ORDER_COL = "label"
sel = cut[cut["label"].isin(CLASSES)].copy()
sel["exists"] = [Path(p).exists() for p in sel["cutout_path"]]

_rolled = sel[sel["fine_order"] != sel["label"]]
if len(_rolled):
    print("rolled up to training labels:")
    for lab, g in _rolled.groupby("label"):
        parts = ", ".join(f"{k} {v}" for k, v in g["fine_order"].value_counts().items())
        print(f"   {lab}: {len(g)} specimens from {parts}")
    print()

print(f"cutouts: {CUTOUT_DIR}")
print(f"output : {OUT_DIR}\n")
print(f"{'class':16s}{'cutouts':>9s}{'on disk':>9s}{'pre-flagged':>13s}")
for c in CLASSES:
    s = sel[sel[ORDER_COL] == c]
    pf = sum(1 for p in s["processid"] if str(p) in PRE_FLAGGED)
    print(f"{c:16s}{len(s):>9d}{int(s['exists'].sum()):>9d}{pf:>13d}")
missing = sel[~sel["exists"]]
if len(missing):
    print(f"\n{len(missing)} cutout file(s) listed in the metadata but absent on disk")
_unknown = [p for p in PRE_FLAGGED if p not in set(sel["processid"].astype(str))]
if _unknown:
    print(f"\npre-flagged ids not found in these classes: {_unknown}")
    print("  check the spelling, or they belong to a class not in CLASSES")


In [ ]:
# ============================================================
# Render every specimen — cutout, and the source photo if reachable
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

_bioscan_cache = {}
def find_source(pid):
    """Locate the original BIOSCAN photograph for a processid, if it is here."""
    if pid in _bioscan_cache: return _bioscan_cache[pid]
    hit = None
    for g in BIOSCAN_GLOBS:
        try:
            hit = next(BASE_DIR.glob(g.format(pid=pid)), None)
        except Exception:
            hit = None
        if hit: break
    _bioscan_cache[pid] = hit
    return hit

def checker(im, sq=14):
    """Composite RGBA over a checkerboard so transparency is visible."""
    w, h = im.size
    d = np.full((h, w, 3), 236, np.uint8)
    yy, xx = np.mgrid[0:h, 0:w]
    d[((yy // sq + xx // sq) % 2) == 0] = 208
    bg = Image.fromarray(d)
    bg.paste(im, (0, 0), im)
    return bg

def stats(path):
    with Image.open(path) as im:
        a = np.array(im.convert("RGBA").split()[-1])
    ys, xs = np.nonzero(a > 10)
    if not len(xs): return None
    w = xs.max()-xs.min()+1; h = ys.max()-ys.min()+1
    return dict(w=int(w), h=int(h), fill=float(len(xs)/(w*h)),
                elong=float(max(w, h)/max(1, min(w, h))))

ROWCOLS = 4
INDEX = {}
for cls in CLASSES:
    s = sel[(sel[ORDER_COL] == cls) & sel["exists"]].reset_index(drop=True)
    if not len(s):
        print(f"{cls}: nothing on disk, skipping"); continue
    cdir = OUT_DIR / cls; cdir.mkdir(exist_ok=True)
    src_found = sum(1 for p in s["processid"] if find_source(str(p)))
    cols = ROWCOLS; rows = int(np.ceil(len(s) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.4*cols, 4.9*rows), squeeze=False)
    for k, ax in enumerate(axes.ravel()):
        if k >= len(s): ax.axis("off"); continue
        r = s.iloc[k]; pid = str(r["processid"])
        with Image.open(r["cutout_path"]) as im:
            ax.imshow(checker(im.convert("RGBA")), interpolation="nearest")
        ax.set_xticks([]); ax.set_yticks([])
        st = stats(r["cutout_path"]) or {}
        flagged = pid in PRE_FLAGGED
        ax.set_title(f"[{k}]  {pid}\n{st.get('w','?')}x{st.get('h','?')}  "
                     f"fill {st.get('fill', float('nan')):.2f}  "
                     f"elong {st.get('elong', float('nan')):.1f}",
                     fontsize=9, color="#A8442A" if flagged else "#171A17",
                     fontweight="bold" if flagged else "normal")
        for sp in ax.spines.values():
            sp.set_edgecolor("#A8442A" if flagged else "#CCCCCC")
            sp.set_linewidth(3 if flagged else 1)
        # full-size copy so a suspect can be opened and zoomed
        with Image.open(r["cutout_path"]) as im:
            checker(im.convert("RGBA")).save(cdir / f"{k:02d}_{pid}.png")
        INDEX[(cls, k)] = pid
    fig.suptitle(f"{cls} — {len(s)} specimens"
                 + (f"   ({src_found} source photos found)" if src_found else
                    "   (source photos not on this machine)"),
                 fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.4)
    fig.savefig(cdir / f"_ALL_{cls}.png", dpi=115, bbox_inches="tight")
    plt.show(); plt.close(fig)
    print(f"{cls}: {len(s)} panels -> {cdir}/_ALL_{cls}.png  (+ full-size copies)")

print(f"\nRed border = already flagged. The [n] in each title is the index to use below.")


## Side by side with the source photograph

Only for specimens whose original BIOSCAN image is on this machine. This is the
panel that separates the two failure modes:

* **source is a fragment** — the photograph itself is a leg or a loose segment.
  The segmentation is correct. Drop the specimen; re-cutting cannot help.
* **source is fine, cutout is wrong** — the pipeline mangled a good photograph.
  Worth re-cutting rather than discarding, which matters when a class has six.

If nothing renders here, the BIOSCAN images are not on this machine and the
judgement has to come from the cutouts alone.


In [ ]:
pairs = []
for cls in CLASSES:
    s = sel[(sel[ORDER_COL] == cls) & sel["exists"]]
    for _, r in s.iterrows():
        src = find_source(str(r["processid"]))
        if src: pairs.append((cls, str(r["processid"]), r["cutout_path"], src))

if not pairs:
    print("No BIOSCAN source images found under", BASE_DIR)
    print("Adjust BIOSCAN_GLOBS if they are elsewhere; otherwise judge from the cutouts.")
else:
    print(f"{len(pairs)} specimens have their source photo available\n")
    for cls in CLASSES:
        p = [x for x in pairs if x[0] == cls]
        if not p: continue
        fig, axes = plt.subplots(len(p), 2, figsize=(9.5, 4.6*len(p)), squeeze=False)
        for i, (_, pid, cpath, spath) in enumerate(p):
            with Image.open(spath) as im:
                axes[i][0].imshow(im.convert("RGB"))
            axes[i][0].set_title(f"{pid} — BIOSCAN source", fontsize=10)
            with Image.open(cpath) as im:
                axes[i][1].imshow(checker(im.convert("RGBA")), interpolation="nearest")
            axes[i][1].set_title("cutout", fontsize=10)
            for a in axes[i]: a.set_xticks([]); a.set_yticks([])
        fig.suptitle(f"{cls} — source vs cutout", fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.97], h_pad=2.0)
        fig.savefig(OUT_DIR / cls / f"_SOURCE_{cls}.png", dpi=115, bbox_inches="tight")
        plt.show(); plt.close(fig)


## Decide

Fill in `DROP` using the `[n]` index from each contact sheet, then run the cell.
Give a reason — it goes into the blocklist and is what you will want in six weeks
when you cannot remember why a specimen went.

Reasons worth distinguishing:

* `fragment` — the photograph is a leg or loose segment, not a whole animal
* `bad-cutout` — good photograph, bad segmentation (re-cut instead of dropping)
* `larva` — a whole animal, but a life stage unlike the rest of the class

Dropping a **larva** is a different decision from dropping a **fragment**. A larva
is correctly labelled and real; it hurts only because the class is too small to
carry two appearances. If Odonata is mostly nymphs, the coherent subset may be the
nymphs and the odd adult is the outlier.


In [ ]:
# Keyed by PROCESSID, not by panel index. An index is only a position in the contact
# sheet, and assuming the flagged panels are the first few silently blocks the wrong
# specimens — which is exactly what happened on the first pass.
DROP = {
    "CISSB6595-23": "fragment",
    "ECORG053-21":  "fragment",
    "ECORG054-21":  "fragment",
    "ECORG684-22":  "fragment",
}

_known = set(sel["processid"].astype(str))
rows = []

# whole sub-orders first
for sub, reason in DROP_SUBORDERS.items():
    hit = sel[sel["fine_order"] == sub]
    if not len(hit):
        print(f"!! sub-order {sub} matched no cutouts — check the spelling; SKIPPED")
        continue
    for _, r in hit.iterrows():
        rows.append({"processid": str(r["processid"]), "order": r["label"],
                     "reason": f"sub-order {sub}: {reason}"})
    print(f"   {sub}: {len(hit)} specimen(s) blocked")
for pid, reason in DROP.items():
    pid = str(pid)
    if pid not in _known:
        print(f"!! {pid} is not in these classes — check the spelling; SKIPPED")
        continue
    rows.append({"processid": pid,
                 "order": sel.loc[sel["processid"].astype(str) == pid, ORDER_COL].iloc[0],
                 "reason": reason})
# PRE_FLAGGED is a REMINDER of what the BIOSCAN site suggested, not a decision.
# Anything you still want dropped must appear in DROP above, so the blocklist only
# ever contains specimens you confirmed against the cutout.
_unconfirmed = [p for p in PRE_FLAGGED
                if p in _known and str(p) not in {r["processid"] for r in rows}]
if _unconfirmed:
    print(f"note: flagged on the BIOSCAN site but NOT in DROP, so kept: {_unconfirmed}\n")

block = pd.DataFrame(rows).drop_duplicates("processid")
block.to_csv(OUT_DIR / "cutout_blocklist.csv", index=False)
print(f"blocklist: {len(block)} specimen(s) -> {OUT_DIR/'cutout_blocklist.csv'}\n")
if len(block): print(block.to_string(index=False))

print(f"\n{'class':16s}{'before':>8s}{'drop':>6s}{'after':>7s}   verdict")
for cls in CLASSES:
    before = int(((sel[ORDER_COL] == cls) & sel["exists"]).sum())
    drop = int((block["order"] == cls).sum()) if len(block) else 0
    after = before - drop
    if after == 0:      v = "CLASS EMPTY — it must be removed from the taxonomy"
    elif after < MIN_KEEP: v = f"below {MIN_KEEP} — still not learnable without more specimens"
    else:               v = "at a workable count"
    kept = sel[(sel["label"] == cls) & sel["exists"] &
               (~sel["processid"].astype(str).isin(set(block["processid"])
                                                   if len(block) else set()))]
    mix = ", ".join(f"{k} {v}" for k, v in kept["fine_order"].value_counts().items())
    print(f"{cls:16s}{before:>8d}{drop:>6d}{after:>7d}   {v}")
    if len(kept) and kept["fine_order"].nunique() > 0 and mix != cls:
        print(f"{'':16s}{'':21s}   remaining: {mix}")

print(f"""
Pruning alone will not rescue these classes — it removes a wrong lesson without
supplying a right one. Before regenerating, check how many of each order BIOSCAN
actually holds against how many were cut; if the pool is much larger, re-cutting
is worth far more than anything decided here.

To apply, filter the generator's specimen pool:

    block = set(pd.read_csv('{OUT_DIR}/cutout_blocklist.csv')['processid'].astype(str))
    all_meta = [m for m in all_meta if str(m['processid']) not in block]
""")
